In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
iris = load_iris(as_frame= True)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [3]:
iris.data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [4]:
iris.target.head()

0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [5]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [6]:
pd.DataFrame(iris.target).value_counts()

target
0         50
1         50
2         50
Name: count, dtype: int64

In [7]:
X = iris.data
y = iris.target

In [8]:
X.corrwith(y)

sepal length (cm)    0.782561
sepal width (cm)    -0.426658
petal length (cm)    0.949035
petal width (cm)     0.956547
dtype: float64

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, stratify= y)

In [10]:
log_reg = LogisticRegression(C= 30, random_state= 42)
log_reg.fit(x_train, y_train)

log_reg.score(x_train, y_train)

0.9833333333333333

In [11]:
predict = log_reg.predict(x_test)

print(log_reg.score(x_test, y_test))

1.0


In [12]:
test = [ i == j for i,j in zip(y_test, predict)]

pd.DataFrame(test).value_counts()

0   
True    30
Name: count, dtype: int64

In [13]:
print(accuracy_score(y_test, predict))
print(confusion_matrix(y_test, predict))
print(classification_report(y_test, predict, target_names= iris.target_names))

1.0
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [14]:
# using Probability
# predict() gives only the winning label, predict_proba() gives how sure the model was.
# Note the separate name: overwriting `predict` would silently break the cells above.

proba = log_reg.predict_proba(x_test)

print(proba.shape)      # (30, 3) -> one column per class, every row sums to 1
pd.DataFrame(proba, columns= iris.target_names).round(3).head()

(30, 3)


,setosa,versicolor,virginica
0,0.999,0.001,0.000
1,0.000,0.322,0.678
2,0.025,0.975,0.000
3,0.017,0.983,0.000
4,1.000,0.000,0.000


In [15]:
# accuracy is 1.0, but which flowers was the model least sure about?
# this is what probabilities buy you that labels cannot.

names = dict(enumerate(iris.target_names))

pd.DataFrame({
    "true": y_test.map(names).values,
    "predicted": iris.target_names[predict],
    "confidence": proba.max(axis= 1).round(3),
}).sort_values("confidence").head()

,true,predicted,confidence
19,virginica,virginica,0.578
25,versicolor,versicolor,0.601
1,virginica,virginica,0.678
23,virginica,virginica,0.940
9,versicolor,versicolor,0.951


In [16]:
# Choosing C properly.
# C is the INVERSE of regularization strength (C ~ 1/alpha in Ridge terms):
# small C = strong penalty = small weights = underfit, large C = weak penalty = overfit.
# C=30 above was picked by watching the test score go up, which quietly makes the
# test set part of training. Choose it with cross-validation on x_train only.

grid = GridSearchCV(LogisticRegression(max_iter= 5000),
                    {"C": np.logspace(-3, 3, 7)},
                    cv= 5)
grid.fit(x_train, y_train)

print("best C        :", grid.best_params_["C"])
print("best CV score :", round(grid.best_score_, 4))
print("test score    :", grid.score(x_test, y_test))   # touched only once, so honest

best C        : 1.0
best CV score : 0.9667
test score    : 0.9666666666666667


In [17]:
# the whole grid, not just the winner: CV score is flat across a wide range of C,
# so the jump to test accuracy 1.0 at C=30 was one flower out of 30, i.e. noise.

pd.DataFrame(grid.cv_results_)[["param_C", "mean_test_score", "std_test_score"]]

,param_C,mean_test_score,std_test_score
0,0.001,0.741667,0.031180
1,0.010,0.866667,0.031180
2,0.100,0.958333,0.000000
3,1.000,0.966667,0.016667
4,10.000,0.966667,0.031180
5,100.000,0.966667,0.031180
6,1000.000,0.958333,0.037268
